# 04 — Customer Churn Definition, Modeling & SHAP Explainability
**Project:** CustomerPulse AI — E-commerce Customer Intelligence & Churn Prediction System

## Part 1 (Stage 7): Leak-Free Temporal Churn Definition
- **Historical Observation Window (`X`):** `2010-12-01` to `2011-09-09` (`282` days)
- **Prediction Cutoff Timestamp:** `2011-09-10 00:00:00`
- **Future Outcome Window (`y`):** `2011-09-10` to `2011-12-09` (`90` days)
- **Right-Censoring Rule:** `953` customers acquired on or after `2011-09-10` are excluded from training/evaluation labels.

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    CHURN_DATASET_PATH,
    CHURN_REPORT_JSON_PATH,
    MODEL_REPORT_JSON_PATH,
    CHURN_PREDICTIONS_PATH,
    SHAP_GLOBAL_IMPORTANCE_PATH,
    SHAP_REPORT_JSON_PATH,
    FIGURES_DIR,
)

with open(CHURN_REPORT_JSON_PATH, "r", encoding="utf-8") as f:
    churn_summary = json.load(f)

pd.DataFrame([churn_summary["cohort_accounting"]]).T.rename(columns={0: "Count / Percentage"})

## Part 2 (Stage 8): Model Training & Holdout Evaluation Comparison

In [ ]:
with open(MODEL_REPORT_JSON_PATH, "r", encoding="utf-8") as f:
    model_report = json.load(f)

print(f"Winning Production Model : {model_report['best_model_name']} ({model_report['model_version']})")
print(f"Optimal OOF Threshold    : {model_report['best_decision_threshold']}")

display(pd.DataFrame(model_report["holdout_test_metrics"]))
display(Image(filename=str(FIGURES_DIR / "06_model_comparison_roc_pr_curves.png")))
display(Image(filename=str(FIGURES_DIR / "07_confusion_matrix_and_threshold_curve.png")))

## Part 3 (Stage 9): Model Explainability with SHAP
### Global Feature Importance & Directional Influence
- **Predictive Association vs. Causation:** SHAP quantifies how each historical behavioral feature shifts the model's predicted churn score relative to the baseline cohort average. These represent empirical predictive associations, not guaranteed causal relationships.

In [ ]:
shap_imp_df = pd.read_csv(SHAP_GLOBAL_IMPORTANCE_PATH)
display(shap_imp_df.head(12))
display(Image(filename=str(FIGURES_DIR / "08_shap_global_importance_bar.png")))
display(Image(filename=str(FIGURES_DIR / "09_shap_directional_beeswarm.png")))

### Local Per-Customer SHAP Explanations (Privacy-Masked Examples)

In [ ]:
with open(SHAP_REPORT_JSON_PATH, "r", encoding="utf-8") as f:
    shap_report = json.load(f)

display(pd.DataFrame(shap_report["individual_customer_examples"]))
display(Image(filename=str(FIGURES_DIR / "10_shap_individual_customer_profiles.png")))